# Let an LLM select relevant tools

The selector sees the tool descriptions and chooses a subset before the main model call. Compare both traces: the catalog before selection and the tools after it. Selection itself is another model request and has a cost.

[Tool selection](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector)

### Candidate selection and guaranteed tools

The selector uses structured output to choose tools for the main model. Its model and prompt can differ from the main agent. `max_tools` caps selected candidates; `always_include` tools are added on top of that cap. Parsing failures can retry, return all candidates, or raise, depending on the policy. The selector changes tool availability rather than executing a tool.

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.tools import inventory
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from model_config import get_model, run_config

In [ ]:
# Execute
before_selection = TraceMiddleware("08_before_selection", pretty=True)
after_selection = TraceMiddleware("08_after_selection", pretty=True)
agent = create_agent(
    model=get_model(), tools=inventory(),
    middleware=[
        before_selection,
        LLMToolSelectorMiddleware(model=get_model("selector"), max_tools=2, always_include=["add"], max_retries=2, on_parsing_failure="error"),
        after_selection,
    ],
)
result = agent.invoke({"messages": [("user", prompt("select"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)
selected_tools = next(e["payload"]["tools"] for e in after_selection.events if e["kind"] == "MODEL REQUEST")
selected_names = [tool.get("function", tool).get("name", tool.get("type")) for tool in selected_tools]
print("Selected tools:", selected_names)
assert "add" in selected_names
assert len(selected_names) <= 3